## 状态分类

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import os
import numpy as np

# 查看生成的数据目录结构
OUT_DIR = "./tahoe_under_10gb_subset"
print("输出目录内容:")
for file in os.listdir(OUT_DIR):
    if file.endswith('.parquet'):
        file_path = os.path.join(OUT_DIR, file)
        file_size = os.path.getsize(file_path) / (1024**2)  # MB
        print(f"  {file} ({file_size:.2f} MB)")

# 读取第一个parquet文件查看数据
parquet_files = [f for f in os.listdir(OUT_DIR) if f.endswith('.parquet')]
if parquet_files:
    first_file = os.path.join(OUT_DIR, parquet_files[0])
    print(f"\n读取文件: {first_file}")
    
    # 读取数据
    df = pq.read_table(first_file).to_pandas()
    
    print(f"数据形状: {df.shape}")
    print(f"\n列名: {list(df.columns)}")
    
    print("\n前5行数据:")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 50)
    print(df.head())
    
    print("\n数据基本信息:")
    print(df.info())
    
    print("\n数值列统计:")
    print(df.describe())
    
    # 查看y值的分布（类别分布）
    print(f"\n类别分布 (y值):")
    y_counts = df['y'].value_counts().sort_index()
    print(y_counts)
    
    # 查看target_gene的分布
    print(f"\n靶基因分布:")
    target_counts = df['target_gene'].value_counts().head(10)
    print(target_counts)
    
    # 查看drug的分布
    print(f"\n药物分布 (前10):")
    drug_counts = df['drug'].value_counts().head(10)
    print(drug_counts)
    
    # 查看cell_line_id的分布
    print(f"\n细胞系分布 (前10):")
    cell_line_counts = df['cell_line_id'].value_counts().head(10)
    print(cell_line_counts)
    
    # 查看gene_ids和counts的结构
    print(f"\n基因数据示例:")
    for i in range(min(3, len(df))):
        print(f"\n样本 {i}:")
        print(f"  y值: {df.iloc[i]['y']}")
        print(f"  靶基因: {df.iloc[i]['target_gene']}")
        print(f"  药物: {df.iloc[i]['drug']}")
        print(f"  基因数量: {len(df.iloc[i]['gene_ids'])}")
        print(f"  前5个基因ID: {df.iloc[i]['gene_ids'][:5]}")
        print(f"  前5个表达量: {df.iloc[i]['counts'][:5]}")
        
        # 检查基因表达量的统计信息
        counts = np.array(df.iloc[i]['counts'])
        print(f"  表达量统计 - 最小值: {counts.min():.2f}, 最大值: {counts.max():.2f}, 平均值: {counts.mean():.2f}")
    
    # 检查数据完整性
    print(f"\n数据完整性检查:")
    print(f"空值统计:")
    print(df.isnull().sum())
    
    # 检查gene_ids和counts的长度是否一致
    lengths_consistent = all(len(row['gene_ids']) == len(row['counts']) for _, row in df.iterrows())
    print(f"基因ID和表达量长度一致: {lengths_consistent}")
    
    # 检查基因数量的分布
    gene_counts = [len(row['gene_ids']) for _, row in df.iterrows()]
    print(f"每个样本的基因数量 - 最小值: {min(gene_counts)}, 最大值: {max(gene_counts)}, 平均值: {np.mean(gene_counts):.1f}")

else:
    print("没有找到parquet文件")

输出目录内容:
  train_0000.parquet (29.17 MB)

读取文件: ./tahoe_under_10gb_subset/train_0000.parquet
数据形状: (12000, 6)

列名: ['gene_ids', 'counts', 'y', 'target_gene', 'drug', 'cell_line_id']

前5行数据:
                                            gene_ids  \
0  [5, 19, 20, 44, 103, 138, 149, 202, 233, 235, ...   
1  [18, 31, 40, 47, 56, 69, 84, 85, 95, 103, 114,...   
2  [39721, 21401, 21437, 17822, 37295, 17902, 112...   
3  [19, 42, 43, 70, 77, 103, 112, 114, 132, 138, ...   
4  [35, 40, 43, 44, 52, 63, 76, 89, 103, 114, 138...   

                                              counts   y target_gene  \
0  [1.0, 2.0, 1.0, 1.0, 2.0, 6.0, 1.0, 1.0, 1.0, ...  45        HRH1   
1  [1.0, 1.0, 1.0, 12.0, 2.0, 1.0, 1.0, 1.0, 1.0,...  75         REN   
2  [553.0, 129.0, 106.0, 49.0, 31.0, 26.0, 24.0, ...  45        HRH1   
3  [1.0, 1.0, 2.0, 1.0, 1.0, 2.0, 2.0, 3.0, 1.0, ...  75         REN   
4  [1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...  75         REN   

          drug cell_line_id  
0  Terfenad


### 12000 samples, 6 features

1. gene_ids (基因ID列表)：每个细胞中表达量最高的基因的标识符
- [5, 19, 20, 44, 103, ...], min=446, max=2048

2. counts (基因表达量列表): 对应gene_ids中每个基因的表达水平
- [1.0, 2.0, 1.0, 1.0, 2.0, ...]
- 与gene_ids一一对应
- 值范围: 1.0-553.0 (浮点数)
- 代表RNA测序的读数计数

3. y (类别标签): 药物靶点基因的数字类别编号
- 范围: 0-92 (共24个类别)
- 分布: 每个类别500个样本，完美平衡
- 示例: 45 → 对应HRH1基因

4. target_gene (靶基因名称)
- 含义: 药物作用的靶点基因的正式名称
- 示例: HRH1, REN, KRAS等
- 特点: 人类可读的基因符号

5. drug (药物名称)
- 含义: 处理细胞所用的化合物名称
- 示例: Terfenadine(特非那定), Aliskiren(阿利吉仑)
- 特点: 每个药物对应一个特定的靶基因

6. cell_line_id (细胞系标识)
- 含义: 实验所用细胞系的唯一标识符
- 格式: CVCL_XXXX (Cellosaurus数据库格式)
- 分布: 不均匀，某些细胞系使用更频繁


### 数据质量
- 无空值: 所有字段都完整
- 数据对齐: gene_ids和counts长度完全一致
- 类别平衡: 每个靶点类别正好500个样本

### 生物学意义

1. 实验设计: 不同药物处理不同细胞系，观察基因表达变化
2. 任务类型: 多分类问题 - 根据基因表达谱预测药物靶点
3. 数据规模: 中等规模，适合机器学习模型训练

### 技术细节
- 基因数量可变: 由于topk_by_counts函数，每个样本保留的基因数不同
- 表达量范围: 大部分基因低表达(平均值1.4-2.46)，少数高表达
- 数据预处理: 已过滤掉多靶点药物，只保留单靶点情况


In [3]:
import os
import pyarrow.parquet as pq

GENE_MD_PATH = os.path.join("./tahoe_small_download", "metadata", "gene_metadata.parquet")
gene_md = pq.read_table(GENE_MD_PATH).to_pandas()

print("gene_metadata columns:", list(gene_md.columns))
print(gene_md.head(3))


gene_metadata columns: ['gene_symbol', 'ensembl_id', 'token_id']
  gene_symbol       ensembl_id  token_id
0      TSPAN6  ENSG00000000003         3
1        TNMD  ENSG00000000005         4
2        DPM1  ENSG00000000419         5


## 废弃

In [1]:
import os
import numpy as np
import pyarrow.parquet as pq
from scgpt.tokenizer.gene_tokenizer import GeneVocab

LOCAL_TAHOE_DIR = "./tahoe_small_download"
GENE_MD_PATH = os.path.join(LOCAL_TAHOE_DIR, "metadata", "gene_metadata.parquet")

# 1) 读取 Tahoe gene_metadata（与你的数据完全一致）
gene_md = pq.read_table(GENE_MD_PATH).to_pandas()

# 明确列名（不再做候选判断）
col_sym = "gene_symbol"
col_ens = "ensembl_id"
col_tid = "token_id"

# 按 token_id 排序，保证 Tahoe gene_id == 行号
gene_md = gene_md.sort_values(col_tid)

# 推荐：用 ensembl_id 对齐 scGPT（gene_symbol 可能有歧义）
tahoe_id_to_gene = gene_md[col_ens].tolist()

# 2) 加载 scGPT vocab
SCGPT_CKPT_DIR = "../reverse/models/scgpt_base/"
VOCAB_JSON = os.path.join(SCGPT_CKPT_DIR, "vocab.json")
VOCAB_PKL  = os.path.join(SCGPT_CKPT_DIR, "model.safetensors")

vocab_path = VOCAB_JSON if os.path.exists(VOCAB_JSON) else VOCAB_PKL
vocab = GeneVocab.from_file(vocab_path)

# 3) 构建 Tahoe token_id -> scGPT vocab_id 映射
map_tahoe_to_scgpt = np.full(len(tahoe_id_to_gene), -1, dtype=np.int32)

for tid, ensg in enumerate(tahoe_id_to_gene):
    if ensg in vocab:
        map_tahoe_to_scgpt[tid] = vocab[ensg]


/home/user/anaconda3/envs/state1/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/user/anaconda3/envs/state1/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/user/anaconda3/envs/state1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 因果分类

In [1]:
print('yes')

yes


In [2]:
import pandas as pd
import os

# 设置数据路径
data_dir = "./tahoe_small_download"  # 或您使用的$OUT路径

# 查看训练数据格式
def inspect_data(file_path, sample_rows=5):
    print(f"=== 查看文件: {file_path} ===")
    
    # 读取Parquet文件
    df = pd.read_parquet(file_path)
    
    # 基本信息
    print(f"数据形状: {df.shape}")
    print(f"列名: {list(df.columns)}")
    print("\n数据类型:")
    print(df.dtypes)
    print(f"\n前{sample_rows}行数据:")
    print(df.head(sample_rows))
    print("\n" + "="*50 + "\n")

# 查看训练数据文件（示例查看第一个文件）
train_files = [f for f in os.listdir(f"{data_dir}/data") if f.endswith('.parquet')]
if train_files:
    inspect_data(f"{data_dir}/data/{train_files[0]}")

# 查看基因元数据
if os.path.exists(f"{data_dir}/metadata/gene_metadata.parquet"):
    inspect_data(f"{data_dir}/metadata/gene_metadata.parquet")

# 查看药物元数据
if os.path.exists(f"{data_dir}/metadata/drug_metadata.parquet"):
    inspect_data(f"{data_dir}/metadata/drug_metadata.parquet")

=== 查看文件: ./tahoe_small_download/data/train-00191-of-03388.parquet ===
数据形状: (28225, 10)
列名: ['genes', 'expressions', 'drug', 'sample', 'BARCODE_SUB_LIB_ID', 'cell_line_id', 'moa-fine', 'canonical_smiles', 'pubchem_cid', 'plate']

数据类型:
genes                 object
expressions           object
drug                  object
sample                object
BARCODE_SUB_LIB_ID    object
cell_line_id          object
moa-fine              object
canonical_smiles      object
pubchem_cid           object
plate                 object
dtype: object

前5行数据:
                                               genes  \
0  [1, 5, 14, 20, 42, 69, 77, 84, 95, 103, 108, 1...   
1  [1, 19, 55, 108, 167, 187, 202, 214, 220, 221,...   
2  [1, 5, 19, 20, 21, 25, 56, 59, 69, 70, 77, 79,...   
3  [1, 19, 42, 45, 56, 81, 109, 112, 114, 128, 16...   
4  [1, 7, 10, 11, 19, 22, 43, 44, 45, 55, 58, 69,...   

                                         expressions  \
0  [-2.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0,...   
1 

In [4]:
import pandas as pd
import numpy as np
import os

def safe_detailed_inspection(file_path):
    print(f"详细分析: {file_path}")
    try:
        df = pd.read_parquet(file_path)
        
        print(f"1. 数据维度: {df.shape}")
        print(f"2. 内存使用: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        
        print("\n3. 各列信息:")
        for i, col in enumerate(df.columns):
            print(f"   {i+1}. {col}: {df[col].dtype}")
            
            try:
                # 安全地检查唯一值数量
                if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
                    unique_count = df[col].nunique()
                    print(f"     唯一值数量: {unique_count}")
                    
                    # 安全地获取示例值
                    sample_values = df[col].head(3).tolist()
                    # 处理可能包含数组的情况
                    sample_str = []
                    for val in sample_values:
                        if isinstance(val, (np.ndarray, list)):
                            sample_str.append(f"array(shape={np.array(val).shape})")
                        else:
                            sample_str.append(str(val))
                    print(f"     示例值: {sample_str}")
                    
                elif pd.api.types.is_numeric_dtype(df[col]):
                    print(f"     范围: {df[col].min():.3f} - {df[col].max():.3f}")
                    print(f"     空值数量: {df[col].isnull().sum()}")
                else:
                    print(f"     数据类型特殊，跳过详细统计")
                    
            except Exception as e:
                print(f"     分析此列时出错: {e}")
                
            print()  # 空行分隔每列信息
                
    except Exception as e:
        print(f"读取文件时出错: {e}")

# 更简单的查看函数
def simple_inspection(file_path):
    print(f"=== 简单查看: {file_path} ===")
    try:
        df = pd.read_parquet(file_path)
        print(f"数据形状: {df.shape}")
        print(f"列名: {list(df.columns)}")
        print("\n前3行数据:")
        print(df.head(3))
        print("\n数据类型:")
        print(df.dtypes)
        print("="*60)
    except Exception as e:
        print(f"错误: {e}")

# 检查数据文件是否存在
data_dir = "./tahoe_small_download"

# 检查目录结构
print("目录结构:")
if os.path.exists(data_dir):
    for root, dirs, files in os.walk(data_dir):
        level = root.replace(data_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # 只显示前5个文件
            print(f"{subindent}{file}")
        if len(files) > 5:
            print(f"{subindent}... 还有 {len(files) - 5} 个文件")
print("\n")

# 查看文件
files_to_check = []

# 检查元数据文件
meta_files = [
    f"{data_dir}/metadata/gene_metadata.parquet",
    f"{data_dir}/metadata/drug_metadata.parquet"
]

# 检查训练数据文件
train_data_dir = f"{data_dir}/data"
if os.path.exists(train_data_dir):
    train_files = [f for f in os.listdir(train_data_dir) if f.endswith('.parquet')]
    if train_files:
        files_to_check.append(f"{train_data_dir}/{train_files[0]}")
    else:
        print("训练数据目录为空")

# 添加元数据文件
for meta_file in meta_files:
    if os.path.exists(meta_file):
        files_to_check.append(meta_file)
    else:
        print(f"文件不存在: {meta_file}")

# 逐个查看文件
for file in files_to_check:
    simple_inspection(file)
    print("\n")

# 如果简单查看成功，再尝试详细分析
print("开始详细分析...")
for file in files_to_check:
    safe_detailed_inspection(file)
    print("\n" + "="*80 + "\n")

目录结构:
tahoe_small_download/
  metadata/
    gene_metadata.parquet
    drug_metadata.parquet
  .cache/
    huggingface/
      .gitignore
      download/
        metadata/
          drug_metadata.parquet.metadata
          drug_metadata.parquet.lock
          gene_metadata.parquet.metadata
          gene_metadata.parquet.lock
          obs_metadata.parquet.lock
          ... 还有 1 个文件
        data/
          train-00032-of-03388.parquet.metadata
          train-00110-of-03388.parquet.metadata
          train-00242-of-03388.parquet.metadata
          train-00010-of-03388.parquet.lock
          train-00080-of-03388.parquet.lock
          ... 还有 655 个文件
  data/
    train-00191-of-03388.parquet
    train-00257-of-03388.parquet
    train-00058-of-03388.parquet
    train-00258-of-03388.parquet
    train-00060-of-03388.parquet
    ... 还有 313 个文件


=== 简单查看: ./tahoe_small_download/data/train-00191-of-03388.parquet ===
数据形状: (28225, 10)
列名: ['genes', 'expressions', 'drug', 'sample', 'BARCODE_SUB_L

## 校验scGPT和tahoe对于基因的编码序号

In [ ]:
import json
import pyarrow.parquet as pq
import pandas as pd

# 1) load gene_metadata
gene_md = pq.read_table("./tahoe_small_download/metadata/gene_metadata.parquet").to_pandas()
# columns: gene_symbol, ensembl_id, token_id

# 2) load scGPT vocab.json
with open("../../gene-perturbation-prediction/model/scGPT/vocab.json", "r") as f:
    vocab = json.load(f)

# vocab.json 可能是 {"token2id":{...}} 或直接 {token:id}
# 兼容两种结构：
if "token2id" in vocab:
    token2id = vocab["token2id"]
else:
    token2id = vocab

# 3) build mapping gene_symbol -> scgpt_id
gene_md["scgpt_id"] = gene_md["gene_symbol"].map(lambda g: token2id.get(g, None))

# 4) check mismatch
valid = gene_md["scgpt_id"].notna()
mismatch = (gene_md.loc[valid, "token_id"].astype(int) != gene_md.loc[valid, "scgpt_id"].astype(int)).sum()
print("genes_in_vocab =", int(valid.sum()))
print("mismatch_count =", int(mismatch))

# 5) if mismatch > 0, build remap: tahoe_token_id -> scgpt_id
remap = dict(zip(
    gene_md.loc[valid, "token_id"].astype(int).tolist(),
    gene_md.loc[valid, "scgpt_id"].astype(int).tolist()
))

# 可选：保存 remap
with open("tahoe_tokenid_to_scgptid.json", "w") as f:
    json.dump(remap, f)


genes_in_vocab = 38913
mismatch_count = 38912


In [ ]:
import json

# 1) label -> gene symbol
lv = json.load(open("label_vocab.json", "r", encoding="utf-8"))
label2gene = {int(k): v for k, v in lv["label2gene"].items()}

# 2) gene symbol -> scGPT id (来自你本地 vocab.json)
with open("./model/scGPT/vocab.json", "r") as f:
    vocab = json.load(f)
token2id = vocab["token2id"] if "token2id" in vocab else vocab

label2target_scgptid = {}
for lab, g in label2gene.items():
    if g in token2id:
        label2target_scgptid[int(lab)] = int(token2id[g])
